# Thực hành Hadoop trên Google Colab

**Huỳnh Thái Học** — Khoa Khoa học dữ liệu trong kinh doanh, Đại học Ngân hàng

Lab triển khai Hadoop ở chế độ **single-node pseudo-distributed** để thực hành:

```text
Cài đặt → Cấu hình HDFS → Khởi động daemon → HDFS Shell
         → MapReduce WordCount → Hadoop Streaming bằng Python → Đối soát
```

> Dữ liệu mô phỏng phục vụ học tập. Mỗi lần Colab cấp runtime mới, cần chạy lại
> từ đầu. Đây là môi trường học lệnh và luồng xử lý, không dùng để đánh giá hiệu
> năng hay mô phỏng độ chịu lỗi của cluster nhiều máy.

## Mục tiêu và sản phẩm

Sau bài này, sinh viên có thể:

1. Phân biệt filesystem local với HDFS namespace.
2. Thao tác `mkdir`, `put`, `ls`, `cat`, `du`, `count`, `get`, `rm` trên HDFS.
3. Giải thích vai trò NameNode, DataNode, block và replication factor.
4. Chạy MapReduce WordCount bằng JAR mẫu.
5. Viết mapper/reducer Python và chạy bằng Hadoop Streaming.
6. Đọc log/counter, xử lý lỗi output đã tồn tại và đối soát kết quả.

**Thời lượng:** 90–120 phút. **Nộp:** notebook đã chạy, ảnh/cell trạng thái HDFS,
kết quả WordCount và phần trả lời câu hỏi cuối bài.

## 0. Quy tắc vận hành

- Chạy theo thứ tự; không format NameNode lần thứ hai khi daemon đang hoạt động.
- Lệnh bắt đầu bằng `!` chạy trong shell nhưng biến `export` không tồn tại qua
  cell khác. Vì vậy notebook đặt biến môi trường bằng `os.environ`.
- HDFS path `/user/student/...` khác local path `/content/...`.
- MapReduce yêu cầu output path **chưa tồn tại**; cell chạy job sẽ chủ động xoá
  output cũ để có thể chạy lại.

In [ ]:
import os
import re
import shutil
import subprocess
from pathlib import Path
import pandas as pd

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
print("Đang chạy trên Colab:", IN_COLAB)
print("Python:", subprocess.check_output(["python3", "--version"], text=True).strip())

## 1. Cài Java và Hadoop

Notebook ghim **Hadoop 3.5.0** thay vì lấy tùy ý bản `latest`. Đây là bản đầu
tiên hỗ trợ đầy đủ Java 17 — phiên bản Java dùng cho daemon. File tải từ Apache
archive để URL không thay đổi khi có phiên bản mới.

Cell này mất vài phút. Nếu đã cài trong runtime hiện tại, cell sẽ bỏ qua tải lại.

In [ ]:
%%bash
set -euo pipefail

HADOOP_VERSION="3.5.0"
HADOOP_ARCHIVE="hadoop-${HADOOP_VERSION}.tar.gz"
HADOOP_URL="https://archive.apache.org/dist/hadoop/common/hadoop-${HADOOP_VERSION}/${HADOOP_ARCHIVE}"

JAVA_MAJOR=$(java -version 2>&1 | awk -F '"' '/version/ {print $2}' | cut -d. -f1 || true)
if [ "${JAVA_MAJOR}" != "17" ]; then
  apt-get update -qq
  apt-get install -y -qq openjdk-17-jdk-headless
  update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
  update-alternatives --set javac /usr/lib/jvm/java-17-openjdk-amd64/bin/javac
fi

if [ ! -x "/content/hadoop-${HADOOP_VERSION}/bin/hadoop" ]; then
  wget -q --show-progress -O "/content/${HADOOP_ARCHIVE}" "${HADOOP_URL}"
  wget -q -O "/content/${HADOOP_ARCHIVE}.sha512" "${HADOOP_URL}.sha512"
  cd /content
  sha512sum -c "${HADOOP_ARCHIVE}.sha512"
  tar -xzf "/content/${HADOOP_ARCHIVE}" -C /content
fi

java -version
/content/hadoop-${HADOOP_VERSION}/bin/hadoop version | head -n 2

### Kiểm tra phiên bản và biến môi trường

Không ghi đè `$HOME`. `JAVA_HOME` được suy ra từ executable Java của runtime.

In [ ]:
HADOOP_VERSION = "3.5.0"
HADOOP_HOME = f"/content/hadoop-{HADOOP_VERSION}"
JAVA_BIN = subprocess.check_output(["bash", "-lc", "readlink -f $(command -v java)"], text=True).strip()
JAVA_HOME = str(Path(JAVA_BIN).parents[1])

os.environ.update({
    "JAVA_HOME": JAVA_HOME,
    "HADOOP_HOME": HADOOP_HOME,
    "HADOOP_CONF_DIR": f"{HADOOP_HOME}/etc/hadoop",
    "HADOOP_COMMON_HOME": HADOOP_HOME,
    "HADOOP_HDFS_HOME": HADOOP_HOME,
    "HADOOP_MAPRED_HOME": HADOOP_HOME,
    "HADOOP_YARN_HOME": HADOOP_HOME,
})
os.environ["PATH"] = f"{HADOOP_HOME}/bin:{HADOOP_HOME}/sbin:" + os.environ["PATH"]

print("JAVA_HOME =", JAVA_HOME)
print("HADOOP_HOME =", HADOOP_HOME)
subprocess.run(["hadoop", "version"], check=True)

## 2. Cấu hình HDFS pseudo-distributed

- `fs.defaultFS`: client mặc định kết nối NameNode tại cổng 9000.
- `dfs.replication=1`: chỉ có một DataNode trong Colab.
- `dfs.namenode.name.dir` và `dfs.datanode.data.dir`: vị trí metadata/block trên
  filesystem local của runtime.

Replication bằng 1 chỉ phù hợp lab; production cần nhiều node, xác thực Kerberos,
phân quyền và vận hành riêng.

In [ ]:
from xml.sax.saxutils import escape

conf_dir = Path(HADOOP_HOME) / "etc/hadoop"
data_root = Path("/content/hadoop-data")
(data_root / "namenode").mkdir(parents=True, exist_ok=True)
(data_root / "datanode").mkdir(parents=True, exist_ok=True)

core_site = '''<?xml version="1.0"?>
<configuration>
  <property><name>fs.defaultFS</name><value>hdfs://localhost:9000</value></property>
  <property><name>hadoop.tmp.dir</name><value>/content/hadoop-tmp</value></property>
</configuration>
'''
hdfs_site = '''<?xml version="1.0"?>
<configuration>
  <property><name>dfs.replication</name><value>1</value></property>
  <property><name>dfs.namenode.name.dir</name><value>file:///content/hadoop-data/namenode</value></property>
  <property><name>dfs.datanode.data.dir</name><value>file:///content/hadoop-data/datanode</value></property>
  <property><name>dfs.permissions.enabled</name><value>false</value></property>
</configuration>
'''
(conf_dir / "core-site.xml").write_text(core_site)
(conf_dir / "hdfs-site.xml").write_text(hdfs_site)

hadoop_env = conf_dir / "hadoop-env.sh"
with hadoop_env.open("a") as stream:
    stream.write(f"\nexport JAVA_HOME={JAVA_HOME}\n")

print((conf_dir / "core-site.xml").read_text())
print((conf_dir / "hdfs-site.xml").read_text())

## 3. Format và khởi động HDFS an toàn

Cell chỉ format nếu chưa có metadata `VERSION`. Daemon được khởi động trực tiếp,
không dùng SSH hay `start-dfs.sh`.

In [ ]:
version_files = list((data_root / "namenode").rglob("VERSION"))
if not version_files:
    print("Format NameNode lần đầu...")
    subprocess.run(["hdfs", "namenode", "-format", "-force", "-nonInteractive"], check=True)
else:
    print("NameNode đã được format — bỏ qua.")

def daemon_running(name):
    result = subprocess.run(["jps"], capture_output=True, text=True, check=True)
    return name in result.stdout

if not daemon_running("NameNode"):
    subprocess.run(["hdfs", "--daemon", "start", "namenode"], check=True)
if not daemon_running("DataNode"):
    subprocess.run(["hdfs", "--daemon", "start", "datanode"], check=True)

subprocess.run(["jps"], check=True)

In [ ]:
# Health checks: đợi HDFS thoát safemode và xác nhận có 1 live DataNode.
import time

for attempt in range(30):
    report = subprocess.run(["hdfs", "dfsadmin", "-report"], capture_output=True, text=True)
    if report.returncode == 0 and "Live datanodes (1)" in report.stdout:
        break
    time.sleep(1)
else:
    raise RuntimeError("HDFS chưa sẵn sàng. Xem log trong $HADOOP_HOME/logs")

subprocess.run(["hdfs", "dfsadmin", "-safemode", "wait"], check=True)
print("HDFS HEALTH CHECK: PASS")
print("\n".join(line for line in report.stdout.splitlines()
                if any(key in line for key in ["Configured Capacity", "DFS Used", "Live datanodes"])))

### Câu hỏi 1

1. NameNode lưu nội dung file hay metadata?
2. DataNode chịu trách nhiệm gì?
3. Vì sao `dfs.replication=3` không phù hợp với Colab một DataNode?
4. Vì sao production cần Kerberos nhưng lab này tắt kiểm tra permission?

## 4. Tạo dữ liệu local

Dữ liệu gồm giao dịch theo vùng và hai file văn bản phục vụ WordCount.

In [ ]:
local_dir = Path("/content/hadoop-lab")
local_dir.mkdir(exist_ok=True)

transactions = '''transaction_id,region,category,amount
GD001,Miền Bắc,Đồ uống,120000
GD002,Miền Nam,Thực phẩm,250000
GD003,Miền Trung,Đồ uống,180000
GD004,Miền Nam,Gia dụng,410000
GD005,Miền Bắc,Thực phẩm,230000
GD006,Miền Nam,Đồ uống,160000
'''
text_1 = '''du lieu tao ra thong tin
thong tin ho tro quyet dinh
hadoop xu ly du lieu lon
'''
text_2 = '''du lieu can chat luong
quyet dinh can bang chung
hadoop mapreduce xu ly song song
'''

(local_dir / "transactions.csv").write_text(transactions, encoding="utf-8")
(local_dir / "lesson1.txt").write_text(text_1, encoding="utf-8")
(local_dir / "lesson2.txt").write_text(text_2, encoding="utf-8")
print("Local files:", [p.name for p in local_dir.iterdir()])

## 5. HDFS Shell cơ bản

Quan sát sự khác nhau:

- `ls /content/hadoop-lab`: local filesystem.
- `hdfs dfs -ls /user/student`: HDFS namespace.

In [ ]:
def run(*args, capture=False):
    result = subprocess.run(args, text=True, capture_output=capture, check=True)
    if capture:
        print(result.stdout)
        return result.stdout

run("hdfs", "dfs", "-mkdir", "-p", "/user/student/input", "/user/student/reference")
run("hdfs", "dfs", "-put", "-f", str(local_dir / "transactions.csv"), "/user/student/reference/")
run("hdfs", "dfs", "-put", "-f", str(local_dir / "lesson1.txt"), str(local_dir / "lesson2.txt"), "/user/student/input/")
run("hdfs", "dfs", "-ls", "-R", "/user/student", capture=True)

In [ ]:
print("NỘI DUNG TRÊN HDFS")
run("hdfs", "dfs", "-cat", "/user/student/reference/transactions.csv", capture=True)

print("DUNG LƯỢNG")
run("hdfs", "dfs", "-du", "-h", "/user/student", capture=True)

print("SỐ THƯ MỤC / FILE / BYTE")
run("hdfs", "dfs", "-count", "-h", "/user/student", capture=True)

### Bài tập 2 — HDFS Shell

Viết lệnh để:

1. Tạo `/user/student/archive`.
2. Sao chép `lesson1.txt` trong HDFS sang archive bằng `-cp`.
3. Đổi tên bản sao thành `lesson1_backup.txt` bằng `-mv`.
4. Tải file backup về `/content/downloaded/` bằng `-get`.
5. Kiểm tra checksum local và HDFS; giải thích vì sao cách biểu diễn có thể khác.
6. Xóa file backup trên HDFS, nhưng không xóa file gốc.

<details><summary><b>Gợi ý lệnh</b></summary>

```bash
hdfs dfs -mkdir -p /user/student/archive
hdfs dfs -cp /user/student/input/lesson1.txt /user/student/archive/
hdfs dfs -mv /user/student/archive/lesson1.txt /user/student/archive/lesson1_backup.txt
mkdir -p /content/downloaded
hdfs dfs -get -f /user/student/archive/lesson1_backup.txt /content/downloaded/
hdfs dfs -checksum /user/student/archive/lesson1_backup.txt
hdfs dfs -rm /user/student/archive/lesson1_backup.txt
```
</details>

## 6. MapReduce WordCount bằng JAR mẫu

Framework đọc các file trong HDFS input, mapper phát sinh `(word, 1)`, shuffle
gom cùng key, reducer cộng số lần xuất hiện và ghi output trở lại HDFS.

Trong notebook này, HDFS chạy pseudo-distributed còn MapReduce dùng local job
runner trong cùng runtime. Muốn thực hành lập lịch phân tán cần cấu hình thêm
YARN hoặc dùng cluster/VM nhiều dịch vụ.

In [ ]:
examples_jar = next(Path(HADOOP_HOME).glob("share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar"))
wc_output = "/user/student/output/wordcount-java"
subprocess.run(["hdfs", "dfs", "-rm", "-r", "-f", wc_output], check=True)
subprocess.run([
    "hadoop", "jar", str(examples_jar), "wordcount",
    "/user/student/input", wc_output,
], check=True)

run("hdfs", "dfs", "-ls", wc_output, capture=True)
java_result = run("hdfs", "dfs", "-cat", f"{wc_output}/part-r-*", capture=True)

### Câu hỏi 3 — Đọc job

1. Vì sao output chứa `_SUCCESS` và `part-r-00000`?
2. Mapper output key/value nào khi đọc từ “du lieu du”?
3. Shuffle/sort thực hiện việc gì?
4. Điều gì xảy ra nếu chạy lại job mà không xoá output cũ?
5. Tìm trong log các counter: input records, map output records và reduce output records.

## 7. Hadoop Streaming với mapper/reducer Python

Streaming cho phép dùng executable đọc `stdin` và ghi `stdout`. Mapper chuẩn
hoá chữ thường, bỏ dấu câu; reducer cộng số đếm theo key đã được Hadoop sắp xếp.

In [ ]:
mapper_code = r'''#!/usr/bin/env python3
import re
import sys

for line in sys.stdin:
    for word in re.findall(r"[a-zA-Z0-9_]+", line.lower()):
        print(f"{word}	1")
'''

reducer_code = r'''#!/usr/bin/env python3
import sys

current_word = None
current_count = 0

for line in sys.stdin:
    word, count = line.rstrip("\n").split("\t", 1)
    count = int(count)
    if word == current_word:
        current_count += count
    else:
        if current_word is not None:
            print(f"{current_word}	{current_count}")
        current_word, current_count = word, count

if current_word is not None:
    print(f"{current_word}	{current_count}")
'''

mapper_path = local_dir / "mapper.py"
reducer_path = local_dir / "reducer.py"
mapper_path.write_text(mapper_code)
reducer_path.write_text(reducer_code)
mapper_path.chmod(0o755); reducer_path.chmod(0o755)
print(mapper_path.read_text())
print(reducer_path.read_text())

### Unit test local trước khi gửi job

Đây là bước quan trọng: kiểm tra mapper/reducer bằng pipe local để tách lỗi Python
khỏi lỗi cấu hình Hadoop.

In [ ]:
local_test = subprocess.run(
    f"cat {local_dir}/lesson*.txt | {mapper_path} | sort | {reducer_path}",
    shell=True, text=True, capture_output=True, check=True,
)
print(local_test.stdout)
assert "hadoop	2" in local_test.stdout
assert "lieu	3" in local_test.stdout
print("LOCAL STREAMING TEST: PASS")

In [ ]:
streaming_jar = next(Path(HADOOP_HOME).glob("share/hadoop/tools/lib/hadoop-streaming-*.jar"))
stream_output = "/user/student/output/wordcount-python"
subprocess.run(["hdfs", "dfs", "-rm", "-r", "-f", stream_output], check=True)

subprocess.run([
    "hadoop", "jar", str(streaming_jar),
    "-D", "mapreduce.job.name=python-wordcount",
    "-files", f"{mapper_path},{reducer_path}",
    "-mapper", "mapper.py", "-reducer", "reducer.py",
    "-input", "/user/student/input",
    "-output", stream_output,
], check=True)

python_result = run("hdfs", "dfs", "-cat", f"{stream_output}/part-*", capture=True)

## 8. Đối soát kết quả

Hai implementation phải cho cùng dictionary WordCount.

In [ ]:
def parse_wordcount(text):
    return dict(line.split("	") for line in text.strip().splitlines())

java_counts = parse_wordcount(java_result)
python_counts = parse_wordcount(python_result)
assert java_counts == python_counts
print("JAVA vs PYTHON STREAMING: KHỚP 100%")
display(pd.DataFrame(sorted(java_counts.items()), columns=["word", "count"]))

## 9. Bài tập tổng hợp — Doanh thu theo vùng

Viết Hadoop Streaming job đọc `transactions.csv`, bỏ header, phát sinh
`region	amount`, sau đó cộng tổng amount theo vùng.

Yêu cầu:

1. Unit test mapper/reducer trên local.
2. Tạo HDFS input riêng, không trộn với file văn bản.
3. Chạy job và lưu vào `/user/student/output/revenue-by-region`.
4. Đối soát bằng Pandas với dữ liệu local.
5. Viết một insight **Quan sát → Ý nghĩa → Hành động**.

<details><summary><b>Gợi ý mapper</b></summary>

```python
import csv, sys
for row in csv.reader(sys.stdin):
    if row and row[0] != "transaction_id":
        print(f"{row[1]}\t{row[3]}")
```

Reducer tương tự WordCount nhưng cộng `amount` thay cho số 1.
</details>

## 10. Chẩn đoán lỗi thường gặp

| Triệu chứng | Kiểm tra | Cách xử lý |
|---|---|---|
| `Connection refused` | `jps`, log NameNode/DataNode | Chạy lại cell khởi động daemon |
| `SafeModeException` | `hdfs dfsadmin -safemode get` | Chờ `-safemode wait`; không tắt cưỡng bức nếu chưa hiểu nguyên nhân |
| `File exists` khi chạy job | `hdfs dfs -ls .../output` | Xoá đúng output path cũ, không xoá input |
| Mapper exit code khác 0 | Unit test local, stderr và YARN/task log | Sửa Python/shebang/quyền executable |
| Không tìm thấy file local | `ls /content/...` | Phân biệt local path với HDFS path |
| DataNode không live | `$HADOOP_HOME/logs` | Kiểm tra JAVA_HOME, data dir và format |

Cell chẩn đoán nhanh:

In [ ]:
print("=== JAVA PROCESSES ===")
subprocess.run(["jps"])
print("\n=== HDFS REPORT ===")
subprocess.run(["hdfs", "dfsadmin", "-report"])
print("\n=== RECENT LOG FILES ===")
for path in sorted((Path(HADOOP_HOME) / "logs").glob("*.log"))[-5:]:
    print(path.name)

## 11. Thu kết quả và dọn runtime

Chạy cell tải kết quả nếu cần. Chỉ chạy cell dừng dịch vụ khi đã hoàn tất lab.

In [ ]:
download_dir = Path("/content/hadoop-results")
if download_dir.exists():
    shutil.rmtree(download_dir)
download_dir.mkdir()
subprocess.run(["hdfs", "dfs", "-get", wc_output, str(download_dir)], check=True)
subprocess.run(["hdfs", "dfs", "-get", stream_output, str(download_dir)], check=True)
print("Đã thu kết quả về:", download_dir)

# Trên Colab, bỏ chú thích nếu muốn tải ZIP:
# shutil.make_archive("/content/hadoop-results", "zip", download_dir)
# from google.colab import files
# files.download("/content/hadoop-results.zip")

In [ ]:
# CHỈ CHẠY SAU KHI HOÀN TẤT BÀI:
# subprocess.run(["hdfs", "--daemon", "stop", "datanode"], check=True)
# subprocess.run(["hdfs", "--daemon", "stop", "namenode"], check=True)

## Tài liệu chính thức

- [Apache Hadoop — Releases](https://hadoop.apache.org/releases.html)
- [Single Node Cluster Setup](https://hadoop.apache.org/docs/r3.5.0/hadoop-project-dist/hadoop-common/SingleCluster.html)
- [HDFS Users Guide](https://hadoop.apache.org/docs/current3/hadoop-project-dist/hadoop-hdfs/HdfsUserGuide.html)
- [MapReduce Tutorial](https://hadoop.apache.org/docs/current3/hadoop-mapreduce-client/hadoop-mapreduce-client-core/MapReduceTutorial.html)
- [Hadoop Streaming](https://hadoop.apache.org/docs/current3/hadoop-streaming/HadoopStreaming.html)

Các bước cấu hình dựa trên tài liệu Apache chính thức. Colab là lựa chọn phục
vụ lớp học; Apache không cung cấp hay bảo đảm môi trường Colab này.